# **PART II: RAG SYSTEM**

Group Members:

-Casucci Leonardo, student ID: 2196383
  
  -Frigo Gianmaria,  student ID: 2196376

Master Program: Computer Engineering

[Short Description and project goal]

In [61]:
from pathlib import Path

CACHE_DIR = Path("artifacts")
RAW_DATA_PATH = CACHE_DIR / "raw_dataset.parquet"
CHUNKS_DF_PATH = CACHE_DIR / "chunks_df.parquet"
EMBEDDINGS_PATH = CACHE_DIR / "embeddings.npy"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

FORCE_RECOMPUTE = False
FORCE_RECOMPUTE_DATA = FORCE_RECOMPUTE
FORCE_RECOMPUTE_CHUNKS = FORCE_RECOMPUTE
FORCE_RECOMPUTE_EMBEDDINGS = FORCE_RECOMPUTE

# **DATASET**

[introduction + summary of your dataset through descriptive statistics]

## Dataset Loading

In [62]:
from datasets import load_dataset
import pandas as pd

if RAW_DATA_PATH.exists() and not FORCE_RECOMPUTE_DATA:
    df = pd.read_parquet(RAW_DATA_PATH)
else:
    dataset = load_dataset(
        "KillerShoaib/Jeffrey-Epstein-Emails-From-Epstein-Files"
    )
    df = dataset["train"].to_pandas()
    df.to_parquet(RAW_DATA_PATH, index=False)


df.head()

,doc_id,subject,preview,from_name,from_email,to,date,body
0,4accfb5f3ed84656e9762740081a4579,cody rudland: You are dead,- Lol good riddance,cody rudland,[redacted],<jeeproject@yahoo.com>,"Aug 13, 2019 6:48 AM",Lol good riddance
1,97d4a52d1df3948368770068262d2aab,Cecilia Steen: With love,"- My dearest Jeffrey, I don't know when you'l...",Cecilia Steen,[redacted],"JE project <jeeproject@yahoo.com>, Jeffrey Eps...","Jul 25, 2019 9:16 AM","My dearest Jeffrey,\nI don't know when you'll ..."
2,HOUSE_OVERSIGHT_016203,Flipboard 10 for Today: 11 questions for Mueller,- Flipboard Interesting stories worth your ti...,Flipboard 10 for Today,editorialstaff@flipboard.com,jeevacation@gmail.com,"Jul 13, 2019 9:06 PM",Flipboard\nInteresting stories worth your time...
3,HOUSE_OVERSIGHT_028801,"Flipboard Week in Review: Alex Acosta resigns,...",- Flipboard Biggest news stories from the pa...,Flipboard Week in Review,editorialstaff@flipboard.com,jeevacation@gmail.com,"Jul 13, 2019 6:48 PM",Flipboard\nBiggest news stories from the past ...
4,HOUSE_OVERSIGHT_029504,[redacted]: [redacted],"- Dear Jeff, I read the news papers today, I ...",[redacted],[redacted],Jeffrey Epstein <jeevacation@gmail.com>,"Jul 10, 2019 8:07 AM","Dear Jeff,\nI read the news papers today, I ca..."


## Dataset Profiling

In [63]:
df.shape
df.info()
df.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 14835 entries, 0 to 14834
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   doc_id      14835 non-null  str  
 1   subject     14835 non-null  str  
 2   preview     14835 non-null  str  
 3   from_name   14835 non-null  str  
 4   from_email  14835 non-null  str  
 5   to          14835 non-null  str  
 6   date        14835 non-null  str  
 7   body        14835 non-null  str  
dtypes: str(8)
memory usage: 19.8 MB


doc_id        0
subject       0
preview       0
from_name     0
from_email    0
to            0
date          0
body          0
dtype: int64

In [64]:
print(f"Number of unique senders: {df['from_email'].nunique()}")
print(f"Number of unique recipients: {df['to'].nunique()}")

Number of unique senders: 455
Number of unique recipients: 1434


In [65]:
df["body"].str.len().describe()

count     14835.000000
mean       1121.424065
std        6707.694938
min           1.000000
25%          84.000000
50%         377.000000
75%         893.000000
max      354403.000000
Name: body, dtype: float64

In [66]:
df["subject"].value_counts().head(10)

subject
jeffrey E.: Re:                                                                                                                                                 1685
J. Epstein: Re:                                                                                                                                                  519
J: Re:                                                                                                                                                           278
J. Epstein: (no subject)                                                                                                                                         220
jeffrey E.: Re: Fwd:                                                                                                                                             217
jeffrey E.: (no subject)                                                                                                                                         198
Je

## Dataset Clean Up

#### Checking for near-empty bodies. They are all reasonable, so we keep them.

In [67]:
df["body_len"] = df["body"].str.len()

short_bodies = df[df["body_len"] < 20]
df.sort_values("body_len")[["doc_id", "subject", "body", "body_len"]].head(20)

,doc_id,subject,body,body_len
459,HOUSE_OVERSIGHT_028576,J: RE: Re:,?,1
1324,HOUSE_OVERSIGHT_032788,jeffrey E.: Re:,?,1
6290,4c8ebdda8fbdccef86a17cee3e82562d,jeffrey E.: Re:,3,1
2169,HOUSE_OVERSIGHT_032706,Kathy Ruemmler: Re:,?,1
5894,HOUSE_OVERSIGHT_029831,[redacted]: Re: Reuters / lawsuit against Jeff...,j,1
501,HOUSE_OVERSIGHT_028589,J: Re:,?,1
1334,HOUSE_OVERSIGHT_032785,jeffrey E.: Re: [redacted],?,1
568,HOUSE_OVERSIGHT_028611,J: Re:,?,1
603,HOUSE_OVERSIGHT_026329,J: Re:,?,1
5757,HOUSE_OVERSIGHT_032873,"Thomas Jr., Landon: Fwd: The book on you...",no,2


#### Normalizing whitespace.

In [68]:
import re

def normalize_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

for col in ["subject", "preview", "from_name", "from_email", "to", "date", "body"]:
    df[col] = df[col].apply(normalize_text)

#### Counting recipients

In [69]:
import re

def count_recipients_by_angle_brackets(to_field):
    if not isinstance(to_field, str) or to_field.strip() == "":
        return 0

    return max(1, len(re.findall(r"<[^<>]+>", to_field)))

df["num_recipients"] = df["to"].apply(count_recipients_by_angle_brackets)
df[["to", "num_recipients"]].sort_values("num_recipients", ascending=False).head(20)

,to,num_recipients
8437,valdemar iodice <valdemar@iodice.com.br>cc isa...,67
1617,"Alan Rogers <[redacted]>, Jeffrey Epstein <jee...",34
6388,"Allen West <[redacted]>, Rafael Bardaji <[reda...",31
12509,"Allen West <[redacted]>, Rafael Bardaji <[reda...",31
7016,"Glenn Young <shakescenes@aol.com>, gm2127 <gm2...",20
5748,Gary Savitzky <GarySavitzky@kamillagordon.oran...,20
6580,Gary Savitzky <rafael.prates@prismabrasil.com....,20
6695,"<sstrohaver@nlg.com>, <jboltax@nextjump.co>, <...",20
5519,"Schecter, Daniel (CC) <[redacted]>, Dunlavey, ...",17
8440,<JEEPROJECT@YAHOO.COM>cc <ehanna@nysgmail.com>...,16


#### Row Deduplication.

Checking if dates are formatted in a consistent way before removing duplicates.

In [70]:
import pandas as pd

df["parsed_date"] = pd.to_datetime(df["date"], errors="coerce")

print("Unparseable dates:", df["parsed_date"].isna().sum())
print("Date range:", df["parsed_date"].min(), "to", df["parsed_date"].max())

C:\Users\gianm\AppData\Local\Temp\ipykernel_14196\230258536.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["parsed_date"] = pd.to_datetime(df["date"], errors="coerce")


Unparseable dates: 0
Date range: 1970-01-01 00:00:00 to 2019-08-13 06:48:00


There is a large number of repeated records appears when using a content-based duplicate key

In [71]:
dedup_key = ["subject", "from_email", "to", "num_recipients", "date", "body"]

print("Full-row duplicates:", df.duplicated().sum())
print("Content-based duplicates:", df.duplicated(subset=dedup_key).sum())

Full-row duplicates: 11
Content-based duplicates: 1236


In [72]:
duplicate_groups = (
    df[df.duplicated(subset=dedup_key, keep=False)]
    .groupby(dedup_key)
    .agg(
        n_rows=("doc_id", "size"),
        doc_ids=("doc_id", lambda x: list(x)),
        n_unique_doc_ids=("doc_id", "nunique"),
        n_unique_previews=("preview", "nunique"),
        n_unique_from_names=("from_name", "nunique")
    )
    .reset_index()
    .sort_values("n_rows", ascending=False)
)

duplicate_groups.head(10)

,subject,from_email,to,num_recipients,date,body,n_rows,doc_ids,n_unique_doc_ids,n_unique_previews,n_unique_from_names
594,jeffrey E.: Re:,jeevacation@gmail.com,[redacted],1,"Sep 8, 2017 11:52 AM",what is your schedule in new york how long wil...,11,"[HOUSE_OVERSIGHT_032622, HOUSE_OVERSIGHT_02366...",11,10,1
595,jeffrey E.: Re:,jeevacation@gmail.com,[redacted],1,"Sep 8, 2017 1:33 PM",I will get larson to call you,11,"[HOUSE_OVERSIGHT_032622, HOUSE_OVERSIGHT_02366...",11,10,1
319,"Thomas Jr., Landon: Re: Saudi money",[redacted],jeffrey E. <jeevacation@gmail.com>,1,"Oct 19, 2016 9:41 AM",Interesting. CEO of big finance form told me t...,9,"[HOUSE_OVERSIGHT_031511, HOUSE_OVERSIGHT_03163...",9,9,1
620,jeffrey E.: Re: Fwd:,[redacted],jeffrey E. <jeevacation@gmail.com>,1,"Sep 19, 2014 7:54 AM",Doesn't look like you are prioritizing your sc...,8,"[HOUSE_OVERSIGHT_023418, HOUSE_OVERSIGHT_02584...",8,8,1
619,jeffrey E.: Re: Fwd:,[redacted],jeffrey E. <jeevacation@gmail.com>,1,"Sep 19, 2014 7:15 AM",Most girls do not have to worry about this crap.,8,"[HOUSE_OVERSIGHT_023418, HOUSE_OVERSIGHT_02584...",8,8,1
618,jeffrey E.: Re: Fwd:,[redacted],jeffrey E. <jeevacation@gmail.com>,1,"Sep 19, 2014 6:15 AM","Agreed, but I need to be prepared to say yes b...",8,"[HOUSE_OVERSIGHT_023418, HOUSE_OVERSIGHT_02584...",8,8,1
611,jeffrey E.: Re: Fwd:,[redacted],jeffrey E. <jeevacation@gmail.com>,1,"Sep 19, 2014 3:02 PM","Uh, yeah. We will now go through a period of s...",8,"[HOUSE_OVERSIGHT_023418, HOUSE_OVERSIGHT_02584...",8,8,1
612,jeffrey E.: Re: Fwd:,[redacted],jeffrey E. <jeevacation@gmail.com>,1,"Sep 19, 2014 3:44 PM",Gov is not jewish,8,"[HOUSE_OVERSIGHT_023418, HOUSE_OVERSIGHT_02584...",8,8,1
609,jeffrey E.: Re: Fwd:,[redacted],jeffrey E. <jeevacation@gmail.com>,1,"Sep 19, 2014 2:57 PM",Reid's guy went down on all 7 counts.,8,"[HOUSE_OVERSIGHT_023418, HOUSE_OVERSIGHT_02584...",8,8,1
610,jeffrey E.: Re: Fwd:,[redacted],jeffrey E. <jeevacation@gmail.com>,1,"Sep 19, 2014 2:59 PM",Yep. And now he is a repeat offender.,8,"[HOUSE_OVERSIGHT_023418, HOUSE_OVERSIGHT_02584...",8,8,1


For the RAG system, repeated identical email content is not useful and may cause the top-k results to contain multiple copies of the same evidence.

The internal document IDs are still useful for source reporting. Therefore, we collapse rows with identical email content into a single row that preserves all `doc_id`s.


In [73]:
df_grouped = (
    df.groupby(dedup_key, dropna=False)
    .agg(
        doc_ids=("doc_id", lambda x: sorted(set(x))),
        n_original_rows=("doc_id", "size"),
        preview=("preview", "first"),
        from_name=("from_name", "first"),
        parsed_date=("parsed_date", "first")
    )
    .reset_index()
)

print("Original rows:", len(df))
print("Rows after content-based grouping:", len(df_grouped))
print("Rows collapsed:", len(df) - len(df_grouped))

Original rows: 14835
Rows after content-based grouping: 13599
Rows collapsed: 1236


## Chunking

In [74]:
def build_header(row):
    # Show up to 5 document IDs to avoid long headers
    doc_ids = ", ".join(row["doc_ids"][:5])
    
    if len(row["doc_ids"]) > 5:
        doc_ids += f", ... ({len(row['doc_ids'])} total)"

    # Truncate recipient list if it's too long
    recipients = str(row["to"])
    if len(recipients) > 200:
        recipients = recipients[:200].rstrip() + " ... (truncated)"

    # Truncate subject if it's too long
    subject = str(row["subject"])
    if len(subject) > 200:
        subject = subject[:200].rstrip() + " ... (truncated)"

    return f"""Subject: {subject}
From: {row['from_name']} <{row['from_email']}>
To: {recipients} of {row['num_recipients']} total recipients
Date: {row['date']}
Source document IDs: {doc_ids}
"""

In [75]:
from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5")

splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=512,
    chunk_overlap=64,
    separators=["\n\n", "\n", ". ", " ", ""]
)

In [76]:
if CHUNKS_DF_PATH.exists() and not FORCE_RECOMPUTE_CHUNKS:
    chunks_df = pd.read_parquet(CHUNKS_DF_PATH)
else:
    chunk_records = []

    for row_idx, row in df_grouped.iterrows():
        header = build_header(row)

        body_chunks = splitter.split_text(row["body"])

        for chunk_idx, body_chunk in enumerate(body_chunks):
            chunk_text = f"""{header}

Body chunk {chunk_idx + 1} of {len(body_chunks)}:
{body_chunk}"""

            chunk_records.append({
                "chunk_id": f"{row_idx}_{chunk_idx}",
                "email_id": row_idx,
                "chunk_index": chunk_idx,
                "num_chunks": len(body_chunks),

                # Metadata
                "doc_ids": row["doc_ids"],
                "n_original_rows": row["n_original_rows"],
                "subject": row["subject"],
                "from_name": row["from_name"],
                "from_email": row["from_email"],
                "to": row["to"],
                "date": row["date"],

                # Text used for retrieval and prompt
                "text_for_embedding": body_chunk,
                "text_for_prompt": chunk_text,
            })

    chunks_df = pd.DataFrame(chunk_records)
    chunks_df.to_parquet(CHUNKS_DF_PATH, index=False)

chunks_df.head()

,chunk_id,email_id,chunk_index,num_chunks,doc_ids,n_original_rows,subject,from_name,from_email,to,date,text_for_embedding,text_for_prompt
0,0_0,0,0,6,[a9daccc94ec91c8d1dc5f4f769c4d722],1,: !!! Re: Love from Prague,,petranemcova@mac.com,Jeffrey Epstein <jeeproject@yahoo.com>,"Jul 29, 2008 4:34 PM",I guess....purity is easy here.. ;-)How is you...,Subject: : !!! Re: Love from Prague\nFrom: <p...
1,0_1,0,1,6,[a9daccc94ec91c8d1dc5f4f769c4d722],1,: !!! Re: Love from Prague,,petranemcova@mac.com,Jeffrey Epstein <jeeproject@yahoo.com>,"Jul 29, 2008 4:34 PM",. And started with special email rules for tim...,Subject: : !!! Re: Love from Prague\nFrom: <p...
2,0_2,0,2,6,[a9daccc94ec91c8d1dc5f4f769c4d722],1,: !!! Re: Love from Prague,,petranemcova@mac.com,Jeffrey Epstein <jeeproject@yahoo.com>,"Jul 29, 2008 4:34 PM",". Epstein jeeproject@yahoo.comSent: Tuesday, M...",Subject: : !!! Re: Love from Prague\nFrom: <p...
3,0_3,0,3,6,[a9daccc94ec91c8d1dc5f4f769c4d722],1,: !!! Re: Love from Prague,,petranemcova@mac.com,Jeffrey Epstein <jeeproject@yahoo.com>,"Jul 29, 2008 4:34 PM",". Epstein"" jeeproject@yahoo.com Date: Wed, 14 ...",Subject: : !!! Re: Love from Prague\nFrom: <p...
4,0_4,0,4,6,[a9daccc94ec91c8d1dc5f4f769c4d722],1,: !!! Re: Love from Prague,,petranemcova@mac.com,Jeffrey Epstein <jeeproject@yahoo.com>,"Jul 29, 2008 4:34 PM",. ELIMINATE. - THE machine is grinding you int...,Subject: : !!! Re: Love from Prague\nFrom: <p...


In [77]:
print(f"Description of number of chunks per email:\n{chunks_df['num_chunks'].describe()}")


Description of number of chunks per email:
count    20912.000000
mean       135.002774
std        287.341037
min          1.000000
25%          1.000000
50%          1.000000
75%          5.000000
max        809.000000
Name: num_chunks, dtype: float64


In [78]:
chunks_df[chunks_df["text_for_embedding"].str.len() < 5]

,chunk_id,email_id,chunk_index,num_chunks,doc_ids,n_original_rows,subject,from_name,from_email,to,date,text_for_embedding,text_for_prompt
1923,361_0,361,0,1,[d48788168076b999d36c4f3ccb75ba2f],1,: Meeting,ehud barak,ehbarak1@gmail.com,jeffrey E. <jeevacation@gmail.com>,"Jul 15, 2014 10:58 PM",Jeff,Subject: : Meeting\nFrom: ehud barak <ehbarak1...
2098,517_0,517,0,1,[HOUSE_OVERSIGHT_028787],1,: Re: B 727 thought,Jeffrey Epstein,jeevacation@gmail.com,[redacted],"Jan 12, 2012 4:57 PM",yes,Subject: : Re: B 727 thought\nFrom: Jeffrey Ep...
3108,679_0,679,0,1,[HOUSE_OVERSIGHT_032204],1,: ass.dads -- 2019/06/01,,jeevacation,[redacted],"Jun 1, 2019 11:49 AM",mon,Subject: : ass.dads -- 2019/06/01\nFrom: <jee...
3109,680_0,680,0,1,[HOUSE_OVERSIGHT_032204],1,: ass.dads -- 2019/06/01,,jeevacation,[redacted],"Jun 1, 2019 12:48 PM",Good,Subject: : ass.dads -- 2019/06/01\nFrom: <jee...
3111,682_0,682,0,1,[HOUSE_OVERSIGHT_032204],1,: ass.dads -- 2019/06/01,,jeevacation,[redacted],"Jun 1, 2019 9:10 PM",Ok,Subject: : ass.dads -- 2019/06/01\nFrom: <jee...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20337,13217_0,13217,0,1,[HOUSE_OVERSIGHT_026459],1,jeffrey E.: Re: [redacted],jeffrey E.,jeevacation@gmail.com,Lesley Groff <[redacted]>,"Nov 16, 2017 11:47 PM",no,Subject: jeffrey E.: Re: [redacted]\nFrom: jef...
20341,13221_0,13221,0,1,[HOUSE_OVERSIGHT_030757],1,jeffrey E.: Re: [redacted],jeffrey E.,jeevacation@gmail.com,Linda Stone <[redacted]>,"Sep 26, 2016 11:34 AM",none,Subject: jeffrey E.: Re: [redacted]\nFrom: jef...
20353,13233_0,13233,0,1,[HOUSE_OVERSIGHT_032663],1,jeffrey E.: Re: [redacted],jeffrey E.,jeevacation@gmail.com,[redacted],"Dec 8, 2017 9:55 PM",Yes,Subject: jeffrey E.: Re: [redacted]\nFrom: jef...
20366,13246_0,13246,0,1,[bb100df32be81498522f9c9c5a4833ce],1,jeffrey E.: Re: a meeting in NY. JE,jeffrey E.,jeevacation@gmail.com,ehud barak <ehbarak1@gmail.com>,"Jul 15, 2014 7:49 PM",230?,Subject: jeffrey E.: Re: a meeting in NY. JE\n...


In [79]:
chunks_len = chunks_df["text_for_embedding"].apply(
    lambda x: len(tokenizer.encode(x, add_special_tokens=True))
)
longest = chunks_len.max()
longest_chunk = chunks_df.iloc[chunks_len.idxmax()]


print(f"Longest chunk with header: {longest} tokens")
print(f"Text of longest chunk:\n{longest_chunk['text_for_prompt']}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (514 > 512). Running this sequence through the model will result in indexing errors


Longest chunk with header: 514 tokens
Text of longest chunk:
Subject: : Fw: Emailing: 667 air ghislaine 008, 667 air ghislaine 009, 667 air ghislaine 010, 667 air ghislaine 011, 667 air ghislaine 006, 667 air ghislaine 007
From:  <lvjet@aol.com>
To: Jeffrey Epstein <jeeproject@yahoo.com> of 1 total recipients
Date: Mar 5, 2008 8:07 PM
Source document IDs: 94f9ed9bac6267a20a85b9a1b82277fb


Body chunk 48 of 742:
XFNO#ZJ[WE_D:Y&NO#ZBZ_WE_D:2W Y_7CG5MKK_KH:H#K5W6SG5+K_KHW\ZH@\TY;B/0O#?&AVWT;^9K@[C%SGW-=[X>XT M*V_W#_,UP%P<R-]:K[(V-'6NV\&C_B5R'UE/\A7$\UW'@W)!+?]=3_(5" MW Y_Q0?^)O/]1_*L?ZUJ^)6SK%S[-C]*R2:N6XBUI_-Y;@_\]%_F*]$U(XL+ MD_\3-OY5YQ8N$NX&/02*?UKNM:O[=+"ZC\U#)LP`#G)/:E%I/4.AP$AR:CCM/S3/7,#_Q]:5S42'F8YZ*HZ_[0_P):L18MCAA]17J8/ ^E>3QG S6Y<>M);IW!#QQ84J/Q_`U4/1C12U!PUY,1W<PZJ@CUH9P_S#O2<^M6W=W)%!S6 MAH5Y'8ZDDTN?+"L#CZ5F\COFFN&)!1@I]<9I-7"YU=YXE66*XCCB(#C:I)Z# M')KF6.23GBH0I[R/^'%/!X[FFM-!-W'IRW3/O31G/8"GQGGI3.35(D5CP<=: MC+R$8"#\6I^/>@"F!&IF+#=Y87TP:G0_.,^M-Q3TQO7IUII";%6\6XAN2Y


# **EMBEDDING INDEX AND RETRIEVAL**

In [80]:
import numpy as np

In [81]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [82]:
def compute_embeddings(chunks_df, embedding_model, batch_size=64):
    
    texts = chunks_df["text_for_embedding"].tolist()

    embeddings = embedding_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    return embeddings.astype("float32")

In [83]:
def load_or_compute_embeddings(
    chunks_df,
    embedding_model,
    embeddings_path=EMBEDDINGS_PATH,
    force_recompute=False
):
    """
    Load embeddings from disk if available and compatible.
    Otherwise compute and save them.
    """
    if embeddings_path.exists() and not force_recompute:
        print(f"Loading embeddings from {embeddings_path}...")
        embeddings = np.load(embeddings_path)

        if embeddings.shape[0] == len(chunks_df):
            return embeddings.astype("float32")

        print(
            "Saved embeddings are incompatible with chunks_df "
            f"({embeddings.shape[0]} embeddings vs {len(chunks_df)} chunks). "
            "Recomputing..."
        )

    else:
        print("No saved embeddings found. Computing embeddings...")

    embeddings = compute_embeddings(chunks_df, embedding_model)

    np.save(embeddings_path, embeddings)
    print(f"Saved embeddings to {embeddings_path}")

    return embeddings

In [84]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5",
    device=device
)

embeddings = load_or_compute_embeddings(
    chunks_df=chunks_df,
    embedding_model=embedding_model,
    force_recompute=FORCE_RECOMPUTE_EMBEDDINGS
)

embeddings.shape

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading embeddings from artifacts\embeddings.npy...


(20912, 768)

## Embedding Index Creation

In [85]:
import faiss

embedding_matrix = embeddings.astype("float32")

index = faiss.IndexFlatIP(embedding_matrix.shape[1])
index.add(embedding_matrix)

## Retrieval

In [86]:
def retrieve(query, k=5):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, k)

    results = chunks_df.iloc[indices[0]].copy()
    results["score"] = scores[0]

    return results

In [87]:
retrieve("What emails mention Bill Clinton private orgy?", k=5)

,chunk_id,email_id,chunk_index,num_chunks,doc_ids,n_original_rows,subject,from_name,from_email,to,date,text_for_embedding,text_for_prompt,score
15994,9254_0,9254,0,1,[HOUSE_OVERSIGHT_030781],1,Steve Bannon: Re:,jeffrey E.,jeevacation@gmail.com,Steve Bannon <[redacted]>,"Feb 25, 2018 11:49 PM",Did you ask him for Hillary's emails ;),Subject: Steve Bannon: Re:\nFrom: jeffrey E. <...,0.710984
18542,11428_0,11428,0,1,[HOUSE_OVERSIGHT_026047],1,jeffrey E.: RE: Re:,jeffrey E.,jeevacation@gmail.com,"Weingarten, Reid","Dec 20, 2016 12:05 PM",So What Does Pizzagate Have To Do With Preside...,Subject: jeffrey E.: RE: Re:\nFrom: jeffrey E....,0.710426
17033,10102_0,10102,0,1,[HOUSE_OVERSIGHT_022187],1,[redacted]: CNN Inquiry,Melanie Hicken,hit-reply@linkedin.com,<[redacted]>,"Oct 13, 2016 8:03 PM","Hi [redacted] I'm a reporter at CNN, and my co...",Subject: [redacted]: CNN Inquiry\nFrom: Melani...,0.710006
20476,13353_0,13353,0,1,[HOUSE_OVERSIGHT_031470],1,jeffrey E.: Re: tomorrows news today,jeffrey E.,jeevacation@gmail.com,Larry Summers <[redacted]>,"May 21, 2016 6:02 AM",http://www.huffingtonpost.ca/2016/05/20/trump-...,Subject: jeffrey E.: Re: tomorrows news today\...,0.708393
20475,13352_0,13352,0,1,[HOUSE_OVERSIGHT_031469],1,jeffrey E.: Re: tomorrows news today,jeffrey E.,jeevacation@gmail.com,,"May 21, 2016 6:02 AM",http://www.huffingtonpost.ca/2016/05/20/trump-...,Subject: jeffrey E.: Re: tomorrows news today\...,0.708393


# **LIBRARIES USED**

[Introduction + documentation of libraries used for sentence embedding, vector store, LLM quantization, etc.]

In [88]:
#code for setup and initialization of libraries

# **PROMPT ENGINEERING**

[Define function that given a query, retrieves significant documents and crafts the query]

[Discussion of design choices taken + results: first bad option discared because..., second option, ..., final option]

In [89]:
#code for the prompt crafting

# **TESTING**

[Choose around 10 to 20 passages from your document repository, and for each passage extract question/answer pairs.]

[To evaluate your system on the micro-test set, use the same chatBot that has extracted the question/answer pairs, and prompt it to act as a judge, passing the question, the gold answer, and the answer you obtain from your system, and ask to provide some score in a given range.]

[Analyse possible errors and inaccuracies, and discuss possible ways of improving your system.]

In [90]:
#code for testing

# **USE OF GENERATIVE AI**

[Describe how AI has been used including examples of used promts]